# Pandas Data Reshaping: `melt()`, `pivot()` and `pivot_table()`

## Learning objectives

By the end, you should be able to:
- Convert a wide DataFrame into a long format with `pd.melt()`.
- Reshape unique row/column combinations with `DataFrame.pivot()`.
- Summarize repeated observations with `DataFrame.pivot_table()`.
- Choose the right function for a data-cleaning or analysis task.

## 1. Setup and sample data

We will use a small student-marks dataset. Each row represents a student's marks in English and Mathematics on a particular day.

In [3]:
import pandas as pd

marks = pd.DataFrame({
    "day": [1, 2, 3, 4, 5, 6, 7],
    "student": ["A", "B", "A", "B", "A", "A", "B"],
    "English": [11, 12, 13, 14, 15, 16, 17],
    "Maths": [13, 12, 15, 17, 18, 13, 16]
})

marks

,day,student,English,Maths
0,1,A,11,13
1,2,B,12,12
2,3,A,13,15
3,4,B,14,17
4,5,A,15,18
5,6,A,16,13
6,7,B,17,16


### Understand the data

- **Wide format:** Different measurements (such as `English` and `Maths`) are stored in separate columns.
- `day` and `student` identify the observation.
- `English` and `Maths` contain the values we may want to reshape.

The same source DataFrame can be reshaped in different ways depending on the task.

## 2. `pd.melt()` — wide format to long format

### What does it do?

`pd.melt()` turns one or more columns into rows. It is useful when values are spread across multiple columns and you want a tidy dataset with one column describing the variable and another containing its value.

### Syntax

```python
pd.melt(
    frame,
    id_vars=None,
    value_vars=None,
    var_name="variable",
    value_name="value"
)
```

- `frame`: DataFrame to reshape.
- `id_vars`: Columns to keep as identifier columns.
- `value_vars`: Columns to unpivot. If omitted, all columns not listed in `id_vars` are used.
- `var_name`: Name for the column that stores the original column names.
- `value_name`: Name for the column that stores the corresponding values.

In [7]:
long_marks = pd.melt(
    marks,
    id_vars=["day", "student"],
    value_vars=["English", "Maths"],
    var_name="subject",
    value_name="score"
)

long_marks

,day,student,subject,score
0,1,A,English,11
1,2,B,English,12
2,3,A,English,13
3,4,B,English,14
4,5,A,English,15
5,6,A,English,16
6,7,B,English,17
7,1,A,Maths,13
8,2,B,Maths,12
9,3,A,Maths,15


### How to read the result

The original `English` and `Maths` columns become two values in the `subject` column. Their marks are placed in `score`. The identifier columns (`day` and `student`) are repeated so each score remains associated with the correct observation.

**Why use it?**
- Preparing data for charts that expect a category column and a value column.
- Combining monthly sales columns such as `Jan`, `Feb`, and `Mar` into a month/value structure.
- Cleaning survey data before grouping or plotting.

### `melt()` with selected value columns

If you only want to reshape specific columns, pass them in `value_vars`. Here, only the marks columns are melted; `day` and `student` remain identifiers.

In [8]:
pd.melt(
    marks,
    id_vars="day",
    value_vars=["English", "Maths"],
    var_name="subject",
    value_name="marks"
)

,day,subject,marks
0,1,English,11
1,2,English,12
2,3,English,13
3,4,English,14
4,5,English,15
5,6,English,16
6,7,English,17
7,1,Maths,13
8,2,Maths,12
9,3,Maths,15


### Check your understanding

1. What would happen if you used `id_vars=["day", "student"]` but left out `value_vars`?
2. Which argument changes the name `variable` to `subject`?
3. In the first example, why do `day` and `student` appear more than once?

**Key point:** `melt()` is often used to make data longer and more tidy.

## 3. `DataFrame.pivot()` — long format to wide format

### What does it do?

`pivot()` reshapes data by using one column to define rows, another to define columns, and optionally a third to fill the resulting cells.

### Syntax

```python
DataFrame.pivot(index=None, columns=None, values=None)
```

- `index`: Column whose unique values should become row labels.
- `columns`: Column whose unique values should become new columns.
- `values`: Column to place inside the table. If omitted, all remaining suitable columns are used.

### Example

Let's create a long-format dataset with one score for each day and subject.

In [10]:
long_scores = pd.DataFrame({
    "day": [1, 1, 2, 2, 3, 3],
    "subject": ["English", "Maths", "English", "Maths", "English", "Maths"],
    "score": [11, 13, 12, 12, 13, 15]
})

long_scores

,day,subject,score
0,1,English,11
1,1,Maths,13
2,2,English,12
3,2,Maths,12
4,3,English,13
5,3,Maths,15


In [11]:
wide_scores = long_scores.pivot(
    index="day",
    columns="subject",
    values="score"
)

wide_scores

subject,English,Maths
day,,
1,11,13
2,12,12
3,13,15


### Interpret the result

- Each unique `day` becomes a row.
- Each unique `subject` becomes a column.
- The `score` values fill the cells.

This is the reverse kind of transformation from `melt()` when the data has a matching structure.

### Important limitation: duplicate combinations

`pivot()` requires every `index` + `columns` combination to identify at most one value. If a day has multiple English scores, Pandas cannot decide which score should fill the cell and raises a `ValueError`.

Run the following cell to see the issue (the error is expected):

In [15]:
duplicate_scores = pd.DataFrame({
    "day": [1, 1, 1],
    "subject": ["English", "English", "Maths"],
    "score": [11, 14, 13]
})

# Expected: ValueError because day=1 and subject=English occurs twice.
# Uncomment the next line to see the error:
# duplicate_scores.pivot(index="day", columns="subject", values="score")

**How to handle duplicates:** If repeated observations should be summarized, use `pivot_table()` and choose an aggregation such as `mean`, `sum`, `min`, or `max`.

**Real-world use cases for `pivot()`:**
- Display one row per date and one column per product.
- Turn category/value records into a report layout.
- Reorganize survey results when each respondent has exactly one value per question.

## 4. `DataFrame.pivot_table()` — reshape and aggregate

### What does it do?

`pivot_table()` creates a spreadsheet-style pivot table. Unlike `pivot()`, it can handle repeated index/column combinations by aggregating them.

### Syntax

```python
DataFrame.pivot_table(
    values=None,
    index=None,
    columns=None,
    aggfunc="mean",
    fill_value=None,
    margins=False
)
```

- `values`: Column(s) to summarize.
- `index`: Grouping field(s) for rows.
- `columns`: Field whose categories become columns.
- `aggfunc`: Calculation applied to each group. Common choices include `"mean"`, `"sum"`, `"count"`, `"min"`, and `"max"`.
- `fill_value`: Replace missing result cells with a chosen value, such as `0`.
- `margins=True`: Add summary totals named `All`. Use the Boolean `True`, not the string `"True"`.

### Example 1: Average score by student and subject

In [16]:
student_scores = pd.DataFrame({
    "student": ["A", "A", "A", "B", "B", "B"],
    "subject": ["English", "English", "Maths", "English", "Maths", "Maths"],
    "score": [70, 90, 80, 60, 70, 90]
})

student_scores

,student,subject,score
0,A,English,70
1,A,English,90
2,A,Maths,80
3,B,English,60
4,B,Maths,70
5,B,Maths,90


In [17]:
average_table = student_scores.pivot_table(
    index="student",
    columns="subject",
    values="score",
    aggfunc="mean"
)

average_table

subject,English,Maths
student,,
A,80.0,80.0
B,60.0,80.0


Student A has two English scores (70 and 90), so the mean is 80. Student B has two Maths scores (70 and 90), so the mean is also 80.

### Example 2: Use a different aggregation function

Change `aggfunc` to `"sum"` to calculate totals instead of averages.

In [18]:
student_scores.pivot_table(
    index="student",
    columns="subject",
    values="score",
    aggfunc="sum",
    fill_value=0
)

subject,English,Maths
student,,
A,160,80
B,60,160


### Example 3: Add row/column grand totals with `margins=True`

The `All` row and column show totals for the selected aggregation. Since this example uses `sum`, the margins are sums.

In [19]:
student_scores.pivot_table(
    index="student",
    columns="subject",
    values="score",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

subject,English,Maths,All
student,,,
A,160,80,240
B,60,160,220
All,220,240,460


### Example 4: Count records

Use `aggfunc="count"` to count non-missing score entries in each student/subject group.

In [20]:
student_scores.pivot_table(
    index="student",
    columns="subject",
    values="score",
    aggfunc="count",
    fill_value=0
)

subject,English,Maths
student,,
A,2,1
B,1,2


### Real-world example: monthly sales report

A business may record multiple sales transactions for each product and month. A pivot table can summarize total sales per product, with months across columns.

In [4]:
sales = pd.DataFrame({
    "product": ["Coffee", "Coffee", "Tea", "Tea", "Coffee", "Tea"],
    "month": ["Jan", "Jan", "Jan", "Feb", "Feb", "Feb"],
    "sales": [1200, 800, 600, 900, 1500, 400]
})

sales.pivot_table(
    index="product",
    columns="month",
    values="sales",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

month,Feb,Jan,All
product,,,
Coffee,1500,2000,3500
Tea,1300,600,1900
All,2800,2600,5400


## 5. `pivot()` vs `pivot_table()` vs `melt()`

| Function | Main purpose | Handles duplicate index/column pairs? | Typical direction |
|---|---|---|---|
| `pd.melt()` | Turn measured columns into rows | Not an aggregation function | Wide → long |
| `DataFrame.pivot()` | Rearrange rows and columns | No; combinations must be unique | Long → wide |
| `DataFrame.pivot_table()` | Rearrange and aggregate values | Yes; uses `aggfunc` | Long → summarized wide |

### Quick decision guide

- Use **`melt()`** when categories are currently column headings and you want them as row values.
- Use **`pivot()`** when each row/column combination has exactly one value.
- Use **`pivot_table()`** when combinations repeat and you need a summary such as mean or sum.

## 6. Common mistakes and tips

1. **Using `pivot()` when keys repeat:** check whether each `index` + `columns` pair is unique; otherwise use `pivot_table()`.
2. **Forgetting `values`:** without it, `pivot()` or `pivot_table()` may reshape multiple remaining columns. Specify `values` when you want one particular measure.
3. **Using `"True"` instead of `True`:** `margins=True` is a Boolean argument.
4. **Confusing `var_name` and `value_name`:** with `melt()`, `var_name` stores original column labels; `value_name` stores their data.
5. **Treating missing cells as zero automatically:** missing combinations can become `NaN`. Use `fill_value=0` only if zero is meaningful for your analysis.
6. **Choosing the wrong aggregation:** `mean` gives averages, `sum` gives totals, and `count` counts non-missing values. Pick the one that matches the business question.

## 7. Practice exercises

Try these yourself before looking at the hints.

1. Melt `marks` but keep only `student` as an identifier. Name the output columns `subject` and `score`.
2. Pivot `long_scores` so `subject` becomes columns and `day` becomes the index.
3. Use `student_scores.pivot_table()` to find the **maximum** score for each student and subject.
4. Use the `sales` DataFrame to find **average** sales by product and month.
5. Explain why `pivot_table()` is appropriate for `student_scores` but a simple `pivot()` is not.

### Hints

- For exercise 1, use `id_vars="student"` and `value_vars=["English", "Maths"]`.
- For exercise 3, use `aggfunc="max"`.
- For exercise 4, use `aggfunc="mean"`.

## 8. Summary

- **`melt()`** converts columns into rows.
- **`pivot()`** reorganizes data when each row/column key combination is unique.
- **`pivot_table()`** reorganizes data and aggregates duplicate combinations.
- Always choose `index`, `columns`, `values`, and `aggfunc` based on the question you want the data to answer.

# Practice exercise Answers

In [34]:
# 1. Melt marks but keep only student as an identifier. Name the output columns subject and score.
import pandas as pd

marks = pd.DataFrame({
    "student": ["Amit", "Priya", "Rahul", "Sneha"],
    "English": [85, 90, 78, 88],
    "Maths": [92, 86, 80, 95]
})

melted_marks = marks.melt(
    id_vars="student",
    value_vars=["English", "Maths"],
    var_name="subject",
    value_name="score"
)

print(melted_marks)

  student  subject  score
0    Amit  English     85
1   Priya  English     90
2   Rahul  English     78
3   Sneha  English     88
4    Amit    Maths     92
5   Priya    Maths     86
6   Rahul    Maths     80
7   Sneha    Maths     95


In [42]:
# 2. Pivot long_scores so subject becomes columns and day becomes the index.
wide_scores = long_scores.pivot(
    index="day",
    columns="subject",
    values="score"
)

print(wide_scores)

subject  English  Maths
day                    
1             11     13
2             12     12
3             13     15


In [43]:
# 3. Find the maximum score for each student and subject
max_scores = student_scores.pivot_table(
    index="student",
    columns="subject",
    values="score",
    aggfunc="max"
)

print(max_scores)

subject  English  Maths
student                
A             90     80
B             60     90


In [44]:
# 4. Find average sales by product and month
average_sales = sales.pivot_table(
    index="product",
    columns="month",
    values="sales",
    aggfunc="mean"
)

print(average_sales)

month       Feb     Jan
product                
Coffee   1500.0  1000.0
Tea       650.0   600.0
